# 🔐 Local Notebook Secret Scanner

Scans notebooks and code files from a **local working directory** for hardcoded secrets,
connection keys, passwords, SAS tokens, and API keys.

| What it does | How |
|---|---|
| Enumerates files | Recursive directory walk |
| Reads notebook + script content | Parses `.ipynb` cells and text files |
| Pattern matching | Regex for common secret patterns |
| Output | Spark DataFrame + optional Delta table |

Use this local mode when you want to scan exported notebooks or source code on disk.

## ⚙️ Configuration

In [ ]:
# --- CONFIGURATION ---

# Local directory to scan. Use absolute path or a path relative to this notebook's working directory.
LOCAL_WORK_DIR = "."

# File types to scan recursively from LOCAL_WORK_DIR
SCAN_EXTENSIONS = [".ipynb", ".py", ".sql", ".scala", ".r", ".txt", ".md"]

# Skip files/folders containing any of these substrings in their relative path
SKIP_PATH_CONTAINS = [".git", "__pycache__"]

# Set to True to save results to a Delta table in the attached lakehouse
SAVE_TO_LAKEHOUSE = False
DELTA_TABLE_NAME = "notebook_secret_scan_results"

# Set to True to print each finding as it's discovered (verbose mode)
VERBOSE = True

## 📁 Local File Helpers

In [ ]:
import json
import re
from pathlib import Path


def resolve_work_dir(path_text):
    p = Path(path_text).expanduser()
    if not p.is_absolute():
        p = (Path.cwd() / p).resolve()
    return p


def should_skip(rel_path_text):
    rel_lower = rel_path_text.lower()
    return any(token.lower() in rel_lower for token in SKIP_PATH_CONTAINS)


def iter_scan_files(root_dir, extensions):
    ext_set = {e.lower() for e in extensions}
    for p in root_dir.rglob("*"):
        if not p.is_file():
            continue
        rel_path_text = p.relative_to(root_dir).as_posix()
        if should_skip(rel_path_text):
            continue
        if p.suffix.lower() in ext_set:
            yield p


def read_notebook_text(nb_path):
    nb_json = json.loads(nb_path.read_text(encoding="utf-8", errors="replace"))
    chunks = []
    for cell in nb_json.get("cells", []):
        src = cell.get("source", [])
        if isinstance(src, list):
            chunks.extend(str(line) for line in src)
            chunks.append("\n")
        elif isinstance(src, str):
            chunks.append(src)
            chunks.append("\n")
    return "".join(chunks)


def read_file_text(file_path):
    if file_path.suffix.lower() == ".ipynb":
        return read_notebook_text(file_path)
    return file_path.read_text(encoding="utf-8", errors="replace")


WORK_DIR = resolve_work_dir(LOCAL_WORK_DIR)
if not WORK_DIR.exists() or not WORK_DIR.is_dir():
    raise ValueError(f"LOCAL_WORK_DIR does not exist or is not a directory: {WORK_DIR}")

print(f"✅ Local scan source ready: {WORK_DIR}")
print(f"✅ Allowed file extensions: {', '.join(SCAN_EXTENSIONS)}")

## 🔍 Secret Detection Patterns

In [ ]:
# Each pattern: (label, compiled regex)
# Patterns are case-insensitive and look for key=value or key:value assignments

SECRET_PATTERNS = [
    # Storage account keys
    ("Storage Account Key",
     re.compile(r'AccountKey\s*=\s*[A-Za-z0-9+/=]{40,}', re.IGNORECASE)),

    # Shared Access Signatures
    ("SAS Token",
     re.compile(r'(\?|&)sig=[A-Za-z0-9%+/=]{20,}', re.IGNORECASE)),
    ("SAS Token (assigned)",
     re.compile(r'["\']https?://[^"\s]+\.blob\.core\.windows\.net[^"\s]*sig=[^"\s]+["\']', re.IGNORECASE)),

    # SQL passwords
    ("SQL Password",
     re.compile(r'(Password|pwd)\s*=\s*[^;\s"]{4,}', re.IGNORECASE)),

    # Cosmos DB keys
    ("Cosmos DB Key",
     re.compile(r'(AccountKey|cosmosdb[_-]?key)\s*[=:]\s*["\']?[A-Za-z0-9+/=]{40,}', re.IGNORECASE)),

    # Generic API keys
    ("API Key",
     re.compile(r'(api[_-]?key|apikey|subscription[_-]?key|Ocp-Apim-Subscription-Key)\s*[=:]\s*["\']?[A-Za-z0-9]{16,}', re.IGNORECASE)),

    # Service principal / client secrets
    ("Client Secret",
     re.compile(r'(client[_-]?secret|app[_-]?secret)\s*[=:]\s*["\']?[A-Za-z0-9~._-]{16,}', re.IGNORECASE)),

    # Connection strings (generic)
    ("Connection String",
     re.compile(r'(connection[_-]?string|conn[_-]?str)\s*[=:]\s*["\'][^"]{20,}["\']', re.IGNORECASE)),

    # Bearer tokens hardcoded
    ("Hardcoded Bearer Token",
     re.compile(r'["\']Bearer\s+eyJ[A-Za-z0-9_-]{20,}', re.IGNORECASE)),

    # Azure Event Hub / Service Bus
    ("Event Hub / Service Bus Key",
     re.compile(r'SharedAccessKey\s*=\s*[A-Za-z0-9+/=]{30,}', re.IGNORECASE)),

    # OpenAI / Azure OpenAI keys
    ("OpenAI API Key",
     re.compile(r'(openai[_-]?api[_-]?key|OPENAI_API_KEY)\s*[=:]\s*["\']?sk-[A-Za-z0-9]{20,}', re.IGNORECASE)),
    ("Azure OpenAI Key",
     re.compile(r'(azure[_-]?openai[_-]?key|AZURE_OPENAI_KEY)\s*[=:]\s*["\']?[A-Fa-f0-9]{32}', re.IGNORECASE)),

    # Generic secret assignment (catches many patterns)
    ("Generic Secret Assignment",
     re.compile(r'(secret|password|passwd|token|credential)\s*[=:]\s*["\'][^"\s]{8,}["\']', re.IGNORECASE)),

    # Spark config with account keys
    ("Spark Config Account Key",
     re.compile(r'spark\.conf\.set\s*\([^)]*account\.key[^)]*\)', re.IGNORECASE)),

    # JDBC URLs with passwords
    ("JDBC Password",
     re.compile(r'jdbc:[a-z]+://[^\s]*password=[^;\s"]{4,}', re.IGNORECASE)),
]

# Lines to ignore (comments explaining secrets, imports, etc.)
IGNORE_PATTERNS = [
    re.compile(r'^\s*#'),       # Python comments
    re.compile(r'^\s*//'),      # JS-style comments
    re.compile(r'<your[_-]'),   # Placeholder prompts
    re.compile(r'\{\{'),        # Template variables
    re.compile(r'<replace'),    # Placeholder prompts
    re.compile(r'PLACEHOLDER', re.IGNORECASE),
    re.compile(r'example\.com', re.IGNORECASE),
]


def is_ignored(line):
    return any(p.search(line) for p in IGNORE_PATTERNS)


def scan_content(text):
    """Scan text and return list of (secret_type, line_number, snippet)."""
    findings = []
    for line_num, line in enumerate(text.split('\n'), start=1):
        if is_ignored(line):
            continue
        for label, pattern in SECRET_PATTERNS:
            if pattern.search(line):
                snippet = line.strip()[:120]
                findings.append((label, line_num, snippet))
                break  # one match per line is enough
    return findings


print(f"✅ Loaded {len(SECRET_PATTERNS)} secret detection patterns")

## 🚀 Scan Local Work Directory

In [ ]:
# --- 1. List candidate files ---
print(f"📂 Scanning local work directory: {WORK_DIR}")
candidate_files = list(iter_scan_files(WORK_DIR, SCAN_EXTENSIONS))
print(f"   Found {len(candidate_files)} candidate files")

# --- 2. Scan each file ---
all_findings = []   # (workspace, notebook, secret_type, line, snippet)
errors = []         # (workspace, notebook, error_msg)
scan_stats = {"files": 0, "findings": 0}

for file_path in candidate_files:
    rel_path = file_path.relative_to(WORK_DIR).as_posix()
    scan_stats["files"] += 1

    if VERBOSE and scan_stats["files"] % 25 == 0:
        print(f"   Progress: {scan_stats['files']}/{len(candidate_files)} files")

    try:
        full_content = read_file_text(file_path)
        findings = scan_content(full_content)

        parent_folder = file_path.parent.relative_to(WORK_DIR).as_posix()
        workspace_name = parent_folder if parent_folder != "." else "(root)"
        notebook_name = rel_path

        for secret_type, line_num, snippet in findings:
            all_findings.append((workspace_name, notebook_name, secret_type, line_num, snippet))
            scan_stats["findings"] += 1
            if VERBOSE:
                print(f"   🚨 {notebook_name} | Line {line_num} | {secret_type}")

    except Exception as e:
        parent_folder = file_path.parent.relative_to(WORK_DIR).as_posix()
        workspace_name = parent_folder if parent_folder != "." else "(root)"
        errors.append((workspace_name, rel_path, str(e)))
        if VERBOSE:
            print(f"   ⚠️  {rel_path}: {str(e)[:120]}")

print(f"\n{'=' * 60}")
print("✅ Scan complete!")
print(f"   Files scanned:  {scan_stats['files']}")
print(f"   Secrets found:  {scan_stats['findings']}")
print(f"   Errors:         {len(errors)}")

## 📊 Results

In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# --- Build findings DataFrame ---
if all_findings:
    schema = StructType([
        StructField("Workspace", StringType(), True),
        StructField("Notebook", StringType(), True),
        StructField("Secret_Type", StringType(), True),
        StructField("Line_Number", IntegerType(), True),
        StructField("Code_Snippet", StringType(), True),
    ])
    df_findings = spark.createDataFrame(
        [Row(Workspace=f[0], Notebook=f[1], Secret_Type=f[2],
             Line_Number=f[3], Code_Snippet=f[4]) for f in all_findings],
        schema=schema
    )
    print(f"🚨 Found {df_findings.count()} hardcoded secrets:\n")
    df_findings.show(100, truncate=False)
else:
    print("✅ No hardcoded secrets found — all clean!")
    df_findings = None

# --- Build errors DataFrame ---
if errors:
    df_errors = spark.createDataFrame(
        [Row(Workspace=e[0], Notebook=e[1], Error=e[2]) for e in errors]
    )
    print(f"\n⚠️  {len(errors)} errors encountered:\n")
    df_errors.show(50, truncate=False)

## 📈 Summary by Secret Type

In [ ]:
if df_findings:
    print("Findings by secret type:\n")
    df_findings.groupBy("Secret_Type").count().orderBy("count", ascending=False).show(truncate=False)

    print("\nFindings by workspace:\n")
    df_findings.groupBy("Workspace").count().orderBy("count", ascending=False).show(truncate=False)

    print("\nTop offending notebooks:\n")
    df_findings.groupBy("Workspace", "Notebook").count().orderBy("count", ascending=False).show(20, truncate=False)

## 💾 Save to Lakehouse (Optional)

In [ ]:
if SAVE_TO_LAKEHOUSE and df_findings:
    from pyspark.sql.functions import current_timestamp

    df_out = df_findings.withColumn("scanned_at", current_timestamp())
    df_out.write.mode("append").format("delta").saveAsTable(DELTA_TABLE_NAME)
    print(f"💾 Saved {df_out.count()} findings to lakehouse table '{DELTA_TABLE_NAME}'")
elif SAVE_TO_LAKEHOUSE:
    print("ℹ️  No findings to save.")
else:
    print("ℹ️  Lakehouse save disabled. Set SAVE_TO_LAKEHOUSE = True to persist results.")